**Name:** Muhammad Bilal  
**RegNo:** FA23-BAI-029

# Setup Mock Files
This cell programmatically creates sample PDF and Word (.docx) files so that all the upcoming code can run successfully without any file errors.

In [6]:
import sys
import subprocess
try:
    import reportlab
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "reportlab"])
try:
    import docx
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx"])

import os
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
import docx

def create_mock_pdf(filename, num_pages=5):
    c = canvas.Canvas(filename, pagesize=letter)
    words_list = [
        "This is the first page of our multi-page PDF document sample.",
        "Page two contains some other text with digits like 12345 and a URL http://example.com.",
        "The third page lists multiple key concepts about natural language processing and computer vision.",
        "This is page number four which has more details for testing word count and keyword search features.",
        "Finally, page five is the last part of this simple mock PDF file with additional sentences."
    ]
    for i in range(num_pages):
        text = words_list[i % len(words_list)]
        c.drawString(100, 750, f"Page {i+1} Title")
        c.drawString(100, 700, text)
        c.showPage()
    c.save()

def create_mock_docx_1(filename):
    doc = docx.Document()
    doc.add_heading("Document A Title", level=1)
    doc.add_paragraph("This is a normal paragraph inside document A for testing.", style="Normal")
    doc.add_heading("Heading Section 1", level=2)
    doc.add_paragraph("Body paragraph with some common words and sentences.", style="Normal")
    doc.add_paragraph("Another body paragraph with more text for statistical count.", style="Normal")
    doc.add_heading("Conclusion", level=2)
    doc.add_paragraph("Short normal final text.", style="Normal")
    doc.save(filename)

def create_mock_docx_2(filename):
    doc = docx.Document()
    doc.add_heading("Document B Title", level=1)
    doc.add_paragraph("This is a normal paragraph inside document B for testing.", style="Normal")
    doc.add_heading("Heading Section 1", level=2)
    doc.add_paragraph("Body paragraph with some common words and sentences.", style="Normal")
    doc.add_paragraph("Unique normal paragraph in Document B only.", style="Normal")
    doc.add_heading("Discussion", level=2)
    doc.add_paragraph("Final short body paragraph.", style="Normal")

    table = doc.add_table(rows=3, cols=3)
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'ID'
    hdr_cells[1].text = 'Name'
    hdr_cells[2].text = 'Value'

    row_cells_1 = table.rows[1].cells
    row_cells_1[0].text = '1'
    row_cells_1[1].text = 'Item A'
    row_cells_1[2].text = '100'

    row_cells_2 = table.rows[2].cells
    row_cells_2[0].text = '2'
    row_cells_2[1].text = 'Item B'
    row_cells_2[2].text = ''

    doc.save(filename)

create_mock_pdf('sample.pdf', 5)
create_mock_docx_1('document_a.docx')
create_mock_docx_2('document_b.docx')
import nltk
nltk.download('stopwords', quiet=True)

True

# Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

This task extracts raw text page-by-page from a PDF file, compiles the data, processes it by removing digits and punctuation, generates token frequencies while filtering standard stopwords, and provides a keyword-based sentence search engine across the extracted text.

In [11]:
import sys
import subprocess
try:
    import PyPDF2
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "PyPDF2"])

import re
import PyPDF2

pages_text = []
with open('sample.pdf', 'rb') as f:
    reader = PyPDF2.PdfReader(f)
    for page in reader.pages:
        pages_text.append(page.extract_text())

print("Total pages:", len(pages_text))

full_text = " ".join(pages_text)
clean_text = re.sub(r'[^a-zA-Z\s]', '', full_text)
tokens = clean_text.lower().split()
print("Total tokens:", len(tokens))

stopwords = {
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours",
    "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers",
    "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves",
    "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is",
    "are", "was", "were", "be", "been", "being", "have", "has", "had", "having",
    "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because",
    "as", "until", "while", "of", "at", "by", "for", "with", "about", "against",
    "between", "into", "through", "during", "before", "after", "above", "below",
    "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again",
    "further", "then", "once"
}

word_freq = {}
for token in tokens:
    if token not in stopwords:
        word_freq[token] = word_freq.get(token, 0) + 1

sorted_freq = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
print("Top 10 words:", sorted_freq[:10])

keyword = input("Enter search keyword: ")
found = False
for idx, text in enumerate(pages_text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    for sentence in sentences:
        if keyword.lower() in sentence.lower():
            print(f"Page {idx+1}: {sentence.strip()}")
            found = True
if not found:
    print("No matching sentence found for the keyword.")

Total pages: 5
Total tokens: 81
Top 10 words: [('page', 10), ('title', 5), ('pdf', 2), ('first', 1), ('multipage', 1), ('document', 1), ('sample', 1), ('two', 1), ('contains', 1), ('some', 1)]
Enter search keyword: doing
No matching sentence found for the keyword.


# Task 2: Structured Page Metadata Extraction and CSV Export

This task parses a multi-page PDF to construct metadata dictionaries containing raw text, word count, and space-excluded character count. It computes extreme word counts, handles an input-based case-insensitive phrase search with a custom exception, and exports key results using Python's built-in csv engine.

In [12]:
import sys
import subprocess
try:
    import PyPDF2
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "PyPDF2"])

import csv
import PyPDF2

class PhraseNotFoundError(Exception):
    pass

pages_data = []
with open('sample.pdf', 'rb') as f:
    reader = PyPDF2.PdfReader(f)
    for idx, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        words = text.split()
        chars = len(text.replace(" ", "").replace("\n", "").replace("\r", ""))
        pages_data.append({
            "page_number": idx + 1,
            "raw_text": text,
            "word_count": len(words),
            "char_count": chars
        })

print("First 2 entries:", pages_data[:2])

valid_pages = [p for p in pages_data if p["word_count"] > 0]

if valid_pages:
    max_page = max(valid_pages, key=lambda x: x["word_count"])
    min_page = min(valid_pages, key=lambda x: x["word_count"])
    avg_words = sum(p["word_count"] for p in pages_data) / len(pages_data)

    print("Max Word Count Page:", max_page["page_number"])
    print("Max Content:", max_page["raw_text"])
    print("Min Word Count Page:", min_page["page_number"])
    print("Average Words:", round(avg_words, 2))

search_phrase = input("Enter search phrase: ")
found_phrase = False
for p in pages_data:
    count = p["raw_text"].lower().count(search_phrase.lower())
    if count > 0:
        print(f"Page {p['page_number']} contains phrase {count} times.")
        found_phrase = True

if not found_phrase:
    try:
        raise PhraseNotFoundError("Phrase not found in document.")
    except PhraseNotFoundError as e:
        print("Caught exception:", e)

with open('pdf_summary.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])
    for p in pages_data:
        writer.writerow([p["page_number"], p["word_count"], p["char_count"]])

with open('pdf_summary.csv', 'r') as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

First 2 entries: [{'page_number': 1, 'raw_text': 'Page 1 Title\nThis is the first page of our multi-page PDF document sample.\n', 'word_count': 14, 'char_count': 61}, {'page_number': 2, 'raw_text': 'Page 2 Title\nPage two contains some other text with digits like 12345 and a URL http://example.com.\n', 'word_count': 17, 'char_count': 83}]
Max Word Count Page: 4
Max Content: Page 4 Title
This is page number four which has more details for testing word count and keyword search features.

Min Word Count Page: 1
Average Words: 17.4
Enter search phrase: i was doing
Caught exception: Phrase not found in document.
['page_number', 'word_count', 'char_count']
['1', '14', '61']
['2', '17', '83']
['3', '17', '94']
['4', '20', '93']
['5', '19', '86']


# Task 3: PDF Text Preprocessing Pipeline for NLP

This pipeline sanitizes full-text output from a PDF by striping URLs, numbers, non-alphabetic elements, and excessive whitespace. It splits sentences, discards empty ones, and uses nltk's standard corpus to drop short sentences following stopword filtering, saving final elements to a flat file.

In [10]:
import sys
import subprocess
try:
    import PyPDF2
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "PyPDF2"])

import re
import PyPDF2
from nltk.corpus import stopwords

with open('sample.pdf', 'rb') as f:
    reader = PyPDF2.PdfReader(f)
    raw_text = " ".join([page.extract_text() or "" for page in reader.pages])

print("Before cleaning count:", len(raw_text))

step1 = re.sub(r'https?:\/\/\S+|www\.\S+', '', raw_text)
step2 = re.sub(r'\d+', '', step1)
step3 = re.sub(r'[^a-zA-Z\s\.]', '', step2)
cleaned = re.sub(r'\s+', ' ', step3).strip()

print("After cleaning count:", len(cleaned))

lowercased = cleaned.lower()
raw_sentences = re.split(r'[.!?]', lowercased)
stripped_sentences = [s.strip() for s in raw_sentences]
non_empty_sentences = [s for s in stripped_sentences if s]

print("Sentences before cleaning empty lines:", len(raw_sentences))
print("Sentences after cleaning empty lines:", len(non_empty_sentences))

stop_words = set(stopwords.words('english'))
final_sentences = []

for s in non_empty_sentences:
    tokens = s.split()
    filtered = [t for t in tokens if t not in stop_words]
    if len(filtered) >= 5:
        final_sentences.append(" ".join(filtered))

print("Before minimum-token filter:", len(non_empty_sentences))
print("After minimum-token filter:", len(final_sentences))
print("Samples:", final_sentences[:3])

with open('nlp_ready_output.txt', 'w') as f:
    for s in final_sentences:
        f.write(s + "\n")

with open('nlp_ready_output.txt', 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

if lines:
    longest = max(lines, key=len)
    shortest = min(lines, key=len)
    print("Total lines:", len(lines))
    print("Longest sentence:", longest)
    print("Shortest sentence:", shortest)
else:
    print("No sentences saved in output file.")

Before cleaning count: 508
After cleaning count: 465
Sentences before cleaning empty lines: 5
Sentences after cleaning empty lines: 4
Before minimum-token filter: 4
After minimum-token filter: 4
Samples: ['page title first page multipage pdf document sample', 'page title page two contains text digits like url page title third page lists multiple key concepts natural language processing computer vision', 'page title page number four details testing word count keyword search features']
Total lines: 4
Longest sentence: page title page two contains text digits like url page title third page lists multiple key concepts natural language processing computer vision
Shortest sentence: page title first page multipage pdf document sample


# Task 4: DOCX Paragraph Classification and Statistical Analysis

This task extracts all paragraph attributes from a DOCX file to perform grouping into Headings, Body text, and other styles. It compiles descriptive word statistics on qualifying Body paragraphs and organizes styling counts in a descending, nested map.

In [13]:
import docx
import pprint

doc = docx.Document('document_a.docx')
paragraphs_data = []
for p in doc.paragraphs:
    paragraphs_data.append({
        "text": p.text,
        "style_name": p.style.name
    })

unique_styles = set(p["style_name"] for p in paragraphs_data)
print("Total paragraphs:", len(paragraphs_data))

headings = []
body = []
other = []

for p in paragraphs_data:
    text = p["text"].strip()
    style = p["style_name"]
    if style.startswith("Heading"):
        headings.append(p)
    elif style == "Normal" or "Body" in style:
        body.append(p)
    else:
        other.append(p)

print("Headings count:", len(headings))
print("Body count:", len(body))
print("Other count:", len(other))

qualifying_body = []
for p in body:
    words = p["text"].split()
    if len(words) > 5:
        chars = len(p["text"].replace(" ", ""))
        qualifying_body.append({
            "text": p["text"],
            "word_count": len(words),
            "char_count": chars
        })

if qualifying_body:
    max_body = max(qualifying_body, key=lambda x: x["word_count"])
    min_body = min(qualifying_body, key=lambda x: x["word_count"])
    avg_body_words = sum(b["word_count"] for b in qualifying_body) / len(qualifying_body)
    print("Max Word Count Paragraph:", max_body["text"])
    print("Min Word Count Paragraph:", min_body["text"])
    print("Mean Word Count:", round(avg_body_words, 2))

nested_styles = {}
for p in paragraphs_data:
    style = p["style_name"]
    if style not in nested_styles:
        nested_styles[style] = {"count": 0, "texts": []}
    nested_styles[style]["count"] += 1
    nested_styles[style]["texts"].append(p["text"])

sorted_styles = dict(sorted(nested_styles.items(), key=lambda x: x[1]["count"], reverse=True))
for style, data in sorted_styles.items():
    if len(data["texts"]) > 2:
        data["texts"] = data["texts"][:2]

pprint.pprint(sorted_styles)

Total paragraphs: 7
Headings count: 3
Body count: 4
Other count: 0
Max Word Count Paragraph: This is a normal paragraph inside document A for testing.
Min Word Count Paragraph: Body paragraph with some common words and sentences.
Mean Word Count: 9.0
{'Heading 1': {'count': 1, 'texts': ['Document A Title']},
 'Heading 2': {'count': 2, 'texts': ['Heading Section 1', 'Conclusion']},
 'Normal': {'count': 4,
            'texts': ['This is a normal paragraph inside document A for '
                      'testing.',
                      'Body paragraph with some common words and sentences.']}}


# Task 5: DOCX Table Extraction and Pandas Integration

This task extracts all tabular data from a DOCX file, loads the primary table into a Pandas DataFrame for missing-value analysis, isolates paragraphs positioned before the primary table, and exports both segments cleanly to a custom pipe-separated text file.

In [14]:
import docx
import pandas as pd

doc = docx.Document('document_b.docx')
tables_list = []
for table in doc.tables:
    grid = []
    for row in table.rows:
        grid.append([cell.text for cell in row.cells])
    tables_list.append(grid)

print("Total tables found:", len(tables_list))
for idx, t in enumerate(tables_list):
    print(f"Table {idx+1} dimensions: {len(t)}x{len(t[0]) if t else 0}")

table_data = tables_list[0]
header = table_data[0]
rows = table_data[1:]
df = pd.DataFrame(rows, columns=header)
display(df)

for col in df.columns:
    empty_count = sum(1 for val in df[col] if val is None or val.strip() == "")
    print(f"Empty cells in column '{col}': {empty_count}")

pre_table_paragraphs = []
for child in doc.element.body:
    if child.tag.endswith('tbl'):
        break
    if child.tag.endswith('p'):
        p = docx.text.paragraph.Paragraph(child, doc)
        pre_table_paragraphs.append(p.text)

print("Pre-table paragraph count:", len(pre_table_paragraphs))

with open('docx_extracted.txt', 'w') as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for p in pre_table_paragraphs:
        f.write(p + "\n")
    f.write("\n--- TABLE DATA ---\n")
    for row in table_data:
        f.write(" | ".join(row) + "\n")

Total tables found: 1
Table 1 dimensions: 3x3


,ID,Name,Value
0,1,Item A,100
1,2,Item B,


Empty cells in column 'ID': 0
Empty cells in column 'Name': 0
Empty cells in column 'Value': 1
Pre-table paragraph count: 7


# Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

This task loads two different word documents to identify common and unique headings, merges the body texts while filtering duplicates, builds a complete token count excluding standard stopwords, and outputs a statistical report.

In [15]:
import re
import docx
from collections import Counter

doc_a = docx.Document('document_a.docx')
doc_b = docx.Document('document_b.docx')

headings_a = [p.text for p in doc_a.paragraphs if p.style.name.startswith("Heading")]
headings_b = [p.text for p in doc_b.paragraphs if p.style.name.startswith("Heading")]

set_a = set(headings_a)
set_b = set(headings_b)

print("Common headings:", set_a.intersection(set_b))
print("Only A:", set_a.difference(set_b))
print("Only B:", set_b.difference(set_a))

body_a = [p.text for p in doc_a.paragraphs if p.style.name == "Normal"]
body_b = [p.text for p in doc_b.paragraphs if p.style.name == "Normal"]
merged_body = body_a + body_b

unique_paragraphs_map = {}
for p in merged_body:
    key = p.strip().lower()
    if key and key not in unique_paragraphs_map:
        unique_paragraphs_map[key] = p

deduplicated_body = list(unique_paragraphs_map.values())
print("Total body counts before deduplication:", len(merged_body))
print("Total body counts after deduplication:", len(deduplicated_body))

stopwords = {
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours",
    "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers",
    "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves",
    "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is",
    "are", "was", "were", "be", "been", "being", "have", "has", "had", "having",
    "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because",
    "as", "until", "while", "of", "at", "by", "for", "with", "about", "against",
    "between", "into", "through", "during", "before", "after", "above", "below",
    "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again",
    "further", "then", "once"
}

all_tokens = []
for p in deduplicated_body:
    clean = re.sub(r'[^a-zA-Z\s]', '', p).lower()
    all_tokens.extend(clean.split())

filtered_tokens = [t for t in all_tokens if t not in stopwords]
word_counts = Counter(filtered_tokens)
print("Top 20 frequent words:", word_counts.most_common(20))

total_headings_count = len(headings_a) + len(headings_b)
total_vocabulary_size = len(word_counts)

paragraph_word_lengths = [len(p.split()) for p in deduplicated_body]
avg_paragraph_length = sum(paragraph_word_lengths) / len(paragraph_word_lengths) if paragraph_word_lengths else 0
longest_body_paragraph = max(deduplicated_body, key=lambda x: len(x.split())) if deduplicated_body else ""

print("Total headings:", total_headings_count)
print("Unique body paragraphs:", len(deduplicated_body))
print("Vocabulary size:", total_vocabulary_size)
print("Average paragraph length:", round(avg_paragraph_length, 2))
print("Longest paragraph:", longest_body_paragraph)

Common headings: {'Heading Section 1'}
Only A: {'Conclusion', 'Document A Title'}
Only B: {'Document B Title', 'Discussion'}
Total body counts before deduplication: 8
Total body counts after deduplication: 7
Top 20 frequent words: [('paragraph', 6), ('normal', 4), ('document', 3), ('body', 3), ('inside', 2), ('testing', 2), ('text', 2), ('short', 2), ('final', 2), ('b', 2), ('some', 1), ('common', 1), ('words', 1), ('sentences', 1), ('another', 1), ('more', 1), ('statistical', 1), ('count', 1), ('unique', 1), ('only', 1)]
Total headings: 6
Unique body paragraphs: 7
Vocabulary size: 20
Average paragraph length: 7.43
Longest paragraph: This is a normal paragraph inside document A for testing.


# Task 7: Multi-Endpoint API Data Joining and Analysis

This task connects to user and post endpoints, validates connection status, compiles posts by their corresponding creator into nested structures, finds active users, and filters elements using address and company features.

In [16]:
import urllib.request
import json

def fetch_endpoint(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response:
        if response.status != 200:
            raise ConnectionError(f"Bad status: {response.status}")
        return json.loads(response.read().decode('utf-8'))

users = fetch_endpoint('https://jsonplaceholder.typicode.com/users')
posts = fetch_endpoint('https://jsonplaceholder.typicode.com/posts')

joined_data = {}
for user in users:
    user_name = user["name"]
    joined_data[user_name] = {
        "email": user["email"],
        "post_count": 0,
        "titles": [],
        "raw_user": user
    }

for post in posts:
    uid = post["userId"]
    user_obj = next((u for u in users if u["id"] == uid), None)
    if user_obj:
        uname = user_obj["name"]
        joined_data[uname]["post_count"] += 1
        joined_data[uname]["titles"].append(post["title"])

first_three_keys = list(joined_data.keys())[:3]
first_three = {k: joined_data[k] for k in first_three_keys}
print("First 3 users:", first_three)

max_author = max(joined_data.items(), key=lambda x: x[1]["post_count"])
author_name = max_author[0]
author_details = max_author[1]
raw_info = author_details["raw_user"]

print("Max Author Name:", author_name)
print("Email:", author_details["email"])
print("Company:", raw_info["company"]["name"])
for idx, t in enumerate(author_details["titles"]):
    print(f"{idx+1}. {t}")

for name, details in joined_data.items():
    raw = details["raw_user"]
    bs = raw["company"]["bs"].lower()
    city = raw["address"]["city"]
    if "synergies" in bs or len(city) > 8:
        print(f"Match: {name} | City: {city} | Posts: {details['post_count']}")

First 3 users: {'Leanne Graham': {'email': 'Sincere@april.biz', 'post_count': 10, 'titles': ['sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'qui est esse', 'ea molestias quasi exercitationem repellat qui ipsa sit aut', 'eum et est occaecati', 'nesciunt quas odio', 'dolorem eum magni eos aperiam quia', 'magnam facilis autem', 'dolorem dolore est ipsam', 'nesciunt iure omnis dolorem tempora et accusantium', 'optio molestias id quia eum'], 'raw_user': {'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}}, 'Ervin Howell': {'email': 'Shanna@melissa.tv', 'post_count': 10, 'titles': ['et 

# Task 8: To-Do Completion Rate Analysis Across Users

This task extracts completion statuses of user to-do lists from an external endpoint, structures total ratios, evaluates validation criteria, and builds a summary ranking of completion rates.

In [17]:
import urllib.request
import json

req = urllib.request.Request('https://jsonplaceholder.typicode.com/todos', headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as r:
    status = r.status
    body = r.read().decode('utf-8')

if status != 200 or not body:
    print("Validation failure on response status or body.")
    exit()

todos = json.loads(body)
if not isinstance(todos, list) or len(todos) == 0:
    print("Validation error on parsed list.")
    exit()

print("Total todos:", len(todos))

grouped = {}
for item in todos:
    uid = item["userId"]
    if uid not in grouped:
        grouped[uid] = {"total": 0, "completed": 0, "incomplete": 0}
    grouped[uid]["total"] += 1
    if item["completed"]:
        grouped[uid]["completed"] += 1
    else:
        grouped[uid]["incomplete"] += 1

print("Grouped todos:", grouped)

for uid, counts in grouped.items():
    is_valid = counts["completed"] + counts["incomplete"] == counts["total"]
    print(f"User {uid} verification:", is_valid)

rates = {}
for uid, counts in grouped.items():
    rates[uid] = round((counts["completed"] / counts["total"]) * 100, 1)

max_user = max(rates.items(), key=lambda x: x[1])
min_user = min(rates.items(), key=lambda x: x[1])
overall_avg = sum(rates.values()) / len(rates)

print("Highest completion user:", max_user)
print("Lowest completion user:", min_user)
print("Overall average completion:", round(overall_avg, 1))

req_users = urllib.request.Request('https://jsonplaceholder.typicode.com/users', headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req_users) as ru:
    users_list = json.loads(ru.read().decode('utf-8'))

user_lookup = {u["id"]: u["name"] for u in users_list}

report_data = []
for uid, rate in rates.items():
    report_data.append({
        "name": user_lookup.get(uid, f"User {uid}"),
        "total": grouped[uid]["total"],
        "completed": grouped[uid]["completed"],
        "rate": rate
    })

report_data.sort(key=lambda x: x["rate"], reverse=True)

print(f"{'Rank':<5} | {'Name':<25} | {'Total Tasks':<11} | {'Completed':<9} | {'Completion Rate %':<17}")
print("-" * 77)
for rank, item in enumerate(report_data):
    print(f"{rank+1:<5} | {item['name']:<25} | {item['total']:<11} | {item['completed']:<9} | {item['rate']:<17}")

Total todos: 200
Grouped todos: {1: {'total': 20, 'completed': 11, 'incomplete': 9}, 2: {'total': 20, 'completed': 8, 'incomplete': 12}, 3: {'total': 20, 'completed': 7, 'incomplete': 13}, 4: {'total': 20, 'completed': 6, 'incomplete': 14}, 5: {'total': 20, 'completed': 12, 'incomplete': 8}, 6: {'total': 20, 'completed': 6, 'incomplete': 14}, 7: {'total': 20, 'completed': 9, 'incomplete': 11}, 8: {'total': 20, 'completed': 11, 'incomplete': 9}, 9: {'total': 20, 'completed': 8, 'incomplete': 12}, 10: {'total': 20, 'completed': 12, 'incomplete': 8}}
User 1 verification: True
User 2 verification: True
User 3 verification: True
User 4 verification: True
User 5 verification: True
User 6 verification: True
User 7 verification: True
User 8 verification: True
User 9 verification: True
User 10 verification: True
Highest completion user: (5, 60.0)
Lowest completion user: (4, 30.0)
Overall average completion: 45.0
Rank  | Name                      | Total Tasks | Completed | Completion Rate %
---

# Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

This final task flattens complex nested dictionary keys from API responses, applies geographic distance logic over latitude and longitude in Pandas, maps hemisphere subsets, and matches forum posts with comment counts.

In [18]:
import urllib.request
import json
import math
import pandas as pd

def flatten_user(user: dict) -> dict:
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"]
    }

req_users = urllib.request.Request('https://jsonplaceholder.typicode.com/users', headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req_users) as ru:
    users = json.loads(ru.read().decode('utf-8'))

flattened_list = [flatten_user(u) for u in users]
print("Flattened list example:", flattened_list[0])

df_users = pd.DataFrame(flattened_list)
display(df_users)

df_users["lat"] = df_users["lat"].astype(float)
df_users["lng"] = df_users["lng"].astype(float)

df_users["distance_from_origin"] = df_users.apply(lambda r: math.sqrt(r["lat"]**2 + r["lng"]**2), axis=1)
max_dist_user = df_users.loc[df_users["distance_from_origin"].idxmax()]
print("User furthest from origin:", max_dist_user["name"])

north_users = df_users[df_users["lat"] > 0]["name"].tolist()
south_users = df_users[df_users["lat"] < 0]["name"].tolist()
print("Northern Hemisphere:", north_users)
print("Southern Hemisphere:", south_users)

avg_north = df_users[df_users["lat"] > 0]["distance_from_origin"].mean()
avg_south = df_users[df_users["lat"] < 0]["distance_from_origin"].mean()

if avg_north > avg_south:
    print("Northern Hemisphere has higher average distance:", avg_north)
else:
    print("Southern Hemisphere has higher average distance:", avg_south)

req_posts = urllib.request.Request('https://jsonplaceholder.typicode.com/posts', headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req_posts) as rp:
    posts = json.loads(rp.read().decode('utf-8'))

req_comments = urllib.request.Request('https://jsonplaceholder.typicode.com/comments', headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req_comments) as rc:
    comments = json.loads(rc.read().decode('utf-8'))

comment_counts = {}
for c in comments:
    pid = c["postId"]
    comment_counts[pid] = comment_counts.get(pid, 0) + 1

post_ranking = []
for p in posts:
    pid = p["id"]
    title = p["title"][:40]
    count = comment_counts.get(pid, 0)
    post_ranking.append((pid, title, count))

post_ranking.sort(key=lambda x: x[2], reverse=True)
print("Top 5 most commented:", post_ranking[:5])
print("Bottom 5 least commented:", post_ranking[-5:])

Flattened list example: {'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'lat': '-37.3159', 'lng': '81.1496', 'company_name': 'Romaguera-Crona', 'company_catchphrase': 'Multi-layered client-server neural-net'}


,id,name,username,email,city,zipcode,lat,lng,company_name,company_catchphrase
0,1,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution
5,6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,South Christy,23505-1337,-71.4197,71.7478,Considine-Lockman,Synchronised bottom-line interface
6,7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,Howemouth,58804-1099,24.8918,21.8984,Johns Group,Configurable multimedia task-force
7,8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,Aliyaview,45169,-14.3990,-120.7677,Abernathy Group,Implemented secondary concept
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,Bartholomebury,76495-3109,24.6463,-168.8889,Yost and Sons,Switchable contextually-based project
9,10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,Lebsackbury,31428-2261,-38.2386,57.2232,Hoeger LLC,Centralized empowering task-force


User furthest from origin: Glenna Reichert
Northern Hemisphere: ['Patricia Lebsack', 'Kurtis Weissnat', 'Glenna Reichert']
Southern Hemisphere: ['Leanne Graham', 'Ervin Howell', 'Clementine Bauch', 'Chelsey Dietrich', 'Mrs. Dennis Schulist', 'Nicholas Runolfsdottir V', 'Clementina DuBuque']
Northern Hemisphere has higher average distance: 123.58329337656919
Top 5 most commented: [(1, 'sunt aut facere repellat provident occae', 5), (2, 'qui est esse', 5), (3, 'ea molestias quasi exercitationem repell', 5), (4, 'eum et est occaecati', 5), (5, 'nesciunt quas odio', 5)]
Bottom 5 least commented: [(96, 'quaerat velit veniam amet cupiditate aut', 5), (97, 'quas fugiat ut perspiciatis vero provide', 5), (98, 'laboriosam dolor voluptates', 5), (99, 'temporibus sit alias delectus eligendi p', 5), (100, 'at nam consequatur ea labore ea harum', 5)]
